# export_simulation.ipynb

Exports simulation JSONL output to Excel — no judge calls required.
Run this immediately after `run_simulation.ipynb` to inspect raw agent behaviour
before (or instead of) running `run_evaluation.ipynb`.

**Output tabs:**
- `Decisions` — one row per agent × event: intervention, decision, reasoning, retrieved memories
- `Reflections` — one row per reflection fired: agent, tick, insight descriptions
- `Run Config` — one row per run: models, ablation flags, agent list
- `Cost & Latency` — agent token counts and cost from `run_summary` (if logged)

---
## Section 0 — Install & imports

In [ ]:
!pip install -q --disable-pip-version-check --no-warn-script-location \
  'pandas>=2.2' \
  openpyxl

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_PATH = '/content/drive/MyDrive/Spring2026/fgenai/SIMULATION/berkeley-homes-wildfire-agent-simulation'
except ImportError:
    PROJECT_PATH = os.path.abspath('..')   # running locally: one level up from notebooks/

os.chdir(PROJECT_PATH)
print(f'Working directory: {os.getcwd()}')

In [ ]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd

print('Imports OK')

---
## Section 1 — Config

**Only this cell needs to change between runs.**

- `RUNS_DIR`: where simulation JSONLs are stored
- `RUN_LABELS_TO_EXPORT`: set to `None` to export all runs found, or list specific labels

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
RUNS_DIR  = Path('outputs/runs')
EXPORT_DIR = Path('outputs/sim_export')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
EXPORT_FILENAME = EXPORT_DIR / f'simulation_{timestamp}.xlsx'

# ── Filter: which run labels to export ────────────────────────────────────────
# Set to None to export everything found in RUNS_DIR.
# Or list specific labels, e.g.:
#   RUN_LABELS_TO_EXPORT = ['Baseline_Full', 'Budget_Full']
RUN_LABELS_TO_EXPORT = None

print(f'Runs dir:    {RUNS_DIR}')
print(f'Export path: {EXPORT_FILENAME}')

---
## Section 2 — Load simulation JSONLs

In [ ]:
def parse_jsonl(path: Path) -> list:
    entries = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                entries.append(json.loads(line))
    return entries

# Structure: {run_label: {run_config, decisions, reflections, run_summary}}
runs = {}

jsonl_files = sorted(RUNS_DIR.glob('*.jsonl'))
print(f'Found {len(jsonl_files)} JSONL files in {RUNS_DIR}\n')

for path in jsonl_files:
    entries = parse_jsonl(path)
    if not entries:
        continue

    run_config = next((e for e in entries if e.get('entry_type') == 'run_config'), None)
    if not run_config:
        print(f'  SKIP {path.name} — no run_config entry found')
        continue

    run_label = run_config.get('run_label', path.stem)

    if RUN_LABELS_TO_EXPORT and run_label not in RUN_LABELS_TO_EXPORT:
        print(f'  SKIP {path.name} — label {run_label!r} not in RUN_LABELS_TO_EXPORT')
        continue

    decisions   = [e for e in entries if e.get('entry_type') == 'decision']
    reflections = [e for e in entries if e.get('entry_type') == 'reflection']
    run_summary = next((e for e in entries if e.get('entry_type') == 'run_summary'), {})

    runs[run_label] = {
        'run_config':  run_config,
        'decisions':   decisions,
        'reflections': reflections,
        'run_summary': run_summary,
        'source_file': path.name,
    }
    print(f'  Loaded {path.name} → label={run_label!r}, '
          f'{len(decisions)} decisions, {len(reflections)} reflections')

print(f'\n{len(runs)} runs loaded.')

---
## Section 3 — Build dataframes

In [ ]:
decision_rows = []

for run_label, run_data in runs.items():
    for entry in run_data['decisions']:
        retrieved = entry.get('retrieved_memories', [])
        retrieved_texts = ' | '.join(
            f"[{m.get('type','?')} imp={m.get('importance','?')}] {m.get('description','')}"
            for m in retrieved
        )
        decision_rows.append({
            'run_label':              run_label,
            'agent_id':               entry.get('agent_id'),
            'agent_display_name':     entry.get('agent_display_name'),
            'day':                    entry.get('tick'),
            'event_type':             entry.get('event_type'),
            'channel':                entry.get('channel'),
            'intervention':           entry.get('intervention'),
            'decision':               entry.get('decision'),
            'reasoning':              entry.get('reasoning'),
            'retrieved_memory_count': len(retrieved),
            'retrieved_memories':     retrieved_texts,
            'new_memory_ids':         str(entry.get('new_memory_ids', [])),
        })

df_decisions = pd.DataFrame(decision_rows)
print(f'Decisions: {len(df_decisions)} rows')

In [ ]:
reflection_rows = []

for run_label, run_data in runs.items():
    for entry in run_data['reflections']:
        for ref in entry.get('reflections', []):
            reflection_rows.append({
                'run_label':          run_label,
                'agent_id':           entry.get('agent_id'),
                'agent_display_name': entry.get('agent_display_name'),
                'day':                entry.get('tick'),
                'reflection_id':      ref.get('id'),
                'importance':         ref.get('importance'),
                'description':        ref.get('description'),
            })

df_reflections = pd.DataFrame(reflection_rows)
print(f'Reflections: {len(df_reflections)} rows')

In [ ]:
config_rows = []

for run_label, run_data in runs.items():
    rc = run_data['run_config']
    config_rows.append({
        'run_label':       run_label,
        'source_file':     run_data['source_file'],
        'scenario_path':   rc.get('scenario_path'),
        'agents':          ', '.join(rc.get('agents', [])),
        'decision_model':  rc.get('decision_model'),
        'reflection_model': rc.get('reflection_model'),
        'use_memory':      rc.get('use_memory'),
        'use_reflection':  rc.get('use_reflection'),
        'concise_output':  rc.get('concise_output'),
    })

df_config = pd.DataFrame(config_rows)
print(f'Run configs: {len(df_config)} rows')

In [ ]:
cost_rows = []

for run_label, run_data in runs.items():
    rs = run_data.get('run_summary', {})
    rc = run_data['run_config']
    cost_rows.append({
        'run_label':           run_label,
        'agent_model':         rc.get('decision_model', ''),
        'agent_tokens_in':     rs.get('agent_tokens_in', ''),
        'agent_tokens_out':    rs.get('agent_tokens_out', ''),
        'agent_cost_usd':      rs.get('agent_cost_usd', ''),
        'total_cost_usd':      rs.get('total_cost_usd', ''),
        'latency_seconds':     rs.get('latency_seconds', ''),
    })

df_cost = pd.DataFrame(cost_rows)
print(f'Cost rows: {len(df_cost)} rows')

---
## Section 4 — Export to Excel

In [ ]:
with pd.ExcelWriter(EXPORT_FILENAME, engine='openpyxl') as writer:
    df_decisions.to_excel(writer,   sheet_name='Decisions',      index=False)
    df_reflections.to_excel(writer, sheet_name='Reflections',    index=False)
    df_config.to_excel(writer,      sheet_name='Run Config',     index=False)
    df_cost.to_excel(writer,        sheet_name='Cost & Latency', index=False)

    # Auto-size all columns
    for sheet_name in writer.sheets:
        ws = writer.sheets[sheet_name]
        for col in ws.columns:
            max_len = max(
                (len(str(cell.value)) if cell.value is not None else 0 for cell in col),
                default=10
            )
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 80)

print(f'Exported to: {EXPORT_FILENAME}')
print(f'  Decisions tab:      {len(df_decisions)} rows')
print(f'  Reflections tab:    {len(df_reflections)} rows')
print(f'  Run Config tab:     {len(df_config)} rows')
print(f'  Cost & Latency tab: {len(df_cost)} rows')

---
## Section 5 — Quick preview

In [ ]:
print('DECISIONS PER RUN × AGENT')
print('=' * 60)
if not df_decisions.empty:
    print(df_decisions.groupby(['run_label', 'agent_display_name']).size()
          .rename('decision_count').to_string())

print('\nREFLECTIONS PER RUN × AGENT')
print('=' * 60)
if not df_reflections.empty:
    print(df_reflections.groupby(['run_label', 'agent_display_name']).size()
          .rename('reflection_count').to_string())
else:
    print('  No reflections logged (use_reflection may be off, or threshold not reached)')

print('\nCOST & LATENCY')
print('=' * 60)
if not df_cost.empty:
    print(df_cost[['run_label', 'agent_model', 'agent_cost_usd', 'latency_seconds']].to_string(index=False))